# Composio (Python HTTP client)\n\nThis notebook uses the **`composio-client`** package — the typed OpenAPI client Composio ships for REST calls. The umbrella **`composio`** PyPI package is **not** used here because it pulls in native extensions (e.g. via `openai` → `jiter`) that **do not install in Pyodide**.

## What you'll learn\n\n1. Install `composio-client` with `micropip`\n2. Set your API key (placeholder in git — replace at runtime only)\n3. List toolkits and tools from the Composio API\n4. Optionally try `tools.execute` (often needs a connected account)\n5. Plot a small summary chart when data is available

### Credentials (read before running)\n\n- Edit the **next code cell** and replace `<REPLACE_AT_RUNTIME>` with your Composio API key.\n- **Do not** save or commit the notebook after pasting a real key.\n- This example does **not** use the pack KV store or `proxies.yml` header injection — the client sends `x-api-key` from your cell (see [Composio Python client](https://pypi.org/project/composio-client/)).

In [ ]:
COMPOSIO_API_KEY = "<REPLACE_AT_RUNTIME>"\nCOMPOSIO_BASE_URL = "https://backend.composio.dev"

### Setup\n\nRun **once** per fresh kernel. `composio-client` and its dependencies are pure Python wheels.

In [ ]:
import micropip

await micropip.install('composio-client==1.39.0')

In [ ]:
COMPOSIO_API_KEY_PLACEHOLDER = "<REPLACE_AT_RUNTIME>"

def composio_credentials_ready() -> bool:
    k = (COMPOSIO_API_KEY or "").strip()
    return bool(k) and k != COMPOSIO_API_KEY_PLACEHOLDER

client = None
if composio_credentials_ready():
    from composio_client import Composio

    client = Composio(api_key=COMPOSIO_API_KEY.strip(), base_url=COMPOSIO_BASE_URL)
    print("Composio client ready (base_url=", COMPOSIO_BASE_URL, ")")
else:
    print(
        "Skipping Composio client: set COMPOSIO_API_KEY in the credentials cell above. "
        "Bundled examples and CI use the placeholder so API calls are skipped."
    )

In [ ]:
if client is None:
    print("Skipping: toolkits.list (no client).")
else:
    try:
        page = client.toolkits.list(limit=10)
        items = page.items or []
        print(f"toolkits (up to 10): {len(items)} returned")
        for t in items[:10]:
            print(f"  - {t.slug}: {t.name}")
    except Exception as e:
        print("toolkits.list failed:", type(e).__name__, str(e)[:500])

In [ ]:
if client is None:
    print("Skipping: tools.list (no client).")
else:
    try:
        page = client.tools.list(toolkit_slug="github", limit=8)
        items = page.items or []
        print(f"github tools (up to 8): {len(items)} returned")
        for t in items[:8]:
            print(f"  - {t.slug}: {t.name}")
    except Exception as e:
        print("tools.list failed:", type(e).__name__, str(e)[:500])

## Checkpoint\n\nWith a real API key you should see toolkit and tool rows above. With the placeholder, cells print skip messages — that is expected.

In [ ]:
if client is None:
    print("Skipping: tools.execute demo (no client).")
else:
    # Execution usually requires a connected account in Composio; this is a best-effort demo.
    demo_slug = None
    try:
        page = client.tools.list(toolkit_slug="github", limit=25)
        for t in page.items or []:
            if getattr(t, "no_auth", False):
                demo_slug = t.slug
                break
        if demo_slug is None and (page.items or []):
            demo_slug = page.items[0].slug
    except Exception as e:
        print("Could not pick a tool slug:", type(e).__name__, str(e)[:300])
        demo_slug = None
    if not demo_slug:
        print("No tool slug selected; skipping execute.")
    else:
        try:
            out = client.tools.execute(demo_slug, arguments={})
            print("execute", demo_slug, "->", type(out).__name__)
            print(out)
        except Exception as e:
            print("tools.execute failed (expected without auth):", type(e).__name__, str(e)[:500])

In [ ]:
if client is None:
    print("Skipping chart (no client).")
else:
    try:
        import matplotlib.pyplot as plt

        page = client.toolkits.list(limit=12)
        items = page.items or []
        labels = [t.slug for t in items[:12]]
        if not labels:
            print("No toolkits to chart.")
        else:
            values = list(range(1, len(labels) + 1))
            plt.figure(figsize=(8, 4))
            plt.bar(labels, values)
            plt.title("Toolkit sample (ordinal index — demo only)")
            plt.ylabel("order")
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print("chart failed:", type(e).__name__, str(e)[:400])

## Next steps\n\n- Connect accounts in the Composio dashboard, then retry `tools.execute`.\n- Open `Cribl_Python_SDK.ipynb` for Cribl inventory patterns side-by-side.

In [ ]:
# ### Prompt:\n# Suggest the next Composio tool calls to explore based on listed toolkits.\n# Context: paste the printed toolkit/tool lines from above.\n# Requirements:\n# - propose 3 concrete tool slugs\n# - note which need OAuth vs no_auth\n# - keep answers short